In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

import seaborn as sns 
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


# I. The Data & ML Architect

In [2]:
# Data Preparation Pipeline
def load_and_prep_data():
    df = sns.load_dataset('penguins')
    
    # Handle Missing Values
    df_clean = df.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex'])
    
    # Filtering & Aggregation
    agg_df = df_clean.groupby(['species', 'island'])['body_mass_g'].mean().reset_index()
    agg_df.rename(columns={'body_mass_g': 'avg_body_mass_g'}, inplace=True)
    
    # Feature Engineering 
    # Calculate the Bill Ratio 
    df_clean['bill_ratio'] = df_clean['bill_length_mm'] / df_clean['bill_depth_mm']
    
    return df_clean, agg_df

#  Pipeline
penguins_df, penguins_agg = load_and_prep_data()

print(f"Size của data sau khi preprocessing: {penguins_df.shape}")
display(penguins_df.head())

Size của data sau khi preprocessing: (333, 8)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,bill_ratio
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,2.090909
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,2.270115
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,2.238889
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,1.901554
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,1.907767


In [6]:
# K-Means Clustering & PCA Integration
def apply_ml_pipeline(df):

    # Feature Selection (Could custom later)
    features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
    X = df[features]
    
    # StandardScaler 
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    #  K-Means Clustering (Find 3 clusters vì mình đang có 3 loài)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df['ml_cluster'] = kmeans.fit_predict(X_scaled)
    df['ml_cluster'] = df['ml_cluster'].astype(str) # Chuyển sang string để vẽ màu rời rạc
    
    # PCA 
    pca = PCA(n_components=2)
    pca_results = pca.fit_transform(X_scaled)
    
    df['pca_1'] = pca_results[:, 0]
    df['pca_2'] = pca_results[:, 1]
    
    variance_explained = pca.explained_variance_ratio_
    print(f"PCA Variance Explained: PC1 = {variance_explained[0]*100:.2f}%, PC2 = {variance_explained[1]*100:.2f}%")
    
    return df

penguins_ml_df = apply_ml_pipeline(penguins_df.copy())
display(penguins_ml_df[['species', 'ml_cluster', 'pca_1', 'pca_2']].head())

PCA Variance Explained: PC1 = 68.63%, PC2 = 19.45%


,species,ml_cluster,pca_1,pca_2
0,Adelie,0,-1.853593,0.032069
1,Adelie,0,-1.316254,-0.443527
2,Adelie,0,-1.376605,-0.161230
4,Adelie,0,-1.885288,-0.012351
5,Adelie,0,-1.919981,0.817598


# II. The Visualization & Interactivity Lead

## 2.1 Charting

In [7]:
# Bar Chart & Distributions
def plot_core_visuals(df):
    # Bar Chart
    fig_bar = px.histogram(
        df, 
        x="island", 
        color="species", 
        barmode="group",
        title="Penguin Count by Island and Species",
        labels={"island": "Island Name"},
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig_bar.update_layout(bargap=0.2)
    fig_bar.show()

    # Distribution Plot: Phân bố Body Mass theo species
    fig_dist = px.histogram(
        df, 
        x="body_mass_g", 
        color="species", 
        marginal="violin", 
        hover_data=df.columns,
        title="Distribution of Body Mass Across Species",
        opacity=0.7,
        barmode="overlay",
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig_dist.show()

plot_core_visuals(penguins_df)

## 2.2 Dashboard Dynamics

In [8]:
# Interactivity & ML Visualization
def plot_interactive_ml_scatter(df):
    # Tạo biểu đồ Scatter với trục x, y (Đât sẽ là 2 thành phần PCA)
    fig = px.scatter(
        df, 
        x="pca_1", 
        y="pca_2", 
        color="species", 
        symbol="ml_cluster", 
        size="body_mass_g", 
        hover_name="species",
        hover_data={
            "pca_1": False, 
            "pca_2": False,
            "island": True,
            "sex": True,
            "body_mass_g": True,
            "bill_ratio": ":.2f", 
            "ml_cluster": True
        },
        title="PCA & K-Means Clustering of Palmer Penguins (Hover for Profile)",
        labels={"ml_cluster": "K-Means Cluster"},
        color_discrete_sequence=px.colors.qualitative.Bold
    )

    # Sliders/Buttons trực tiếp trong Plotly
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="right",
                x=0.1,
                y=1.15,
                buttons=list([
                    dict(label="All",
                         method="update",
                         args=[{"visible": [True, True, True]}]),
                    dict(label="Male Only",
                         method="update",
                         args=[{"visible": [True if s == 'Male' else False for s in df['sex'].unique()]}]),
                    dict(label="Female Only",
                         method="update",
                         args=[{"visible": [True if s == 'Female' else False for s in df['sex'].unique()]}])
                ]),
            )
        ]
    )

    fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
    fig.show()

plot_interactive_ml_scatter(penguins_ml_df)